In [0]:
pip install langchain_core langchain_community

In [0]:
import langchain_core
import langchain_community

In [0]:
# Databricks notebook source
# DBTITLE 1,Test Entity Extraction
# This notebook helps you test why extraction is returning empty results

"""
Use this to diagnose extraction issues by:
1. Checking what's in your scraped_text
2. Testing extraction with sample text
3. Improving the prompt if needed
"""

# COMMAND ----------

# DBTITLE 1,Setup
policy_catalog = "gklick_catalog"
policy_schema = "aipolicyassistant"

# COMMAND ----------

# DBTITLE 1,Check Source Documents
print("=== Checking Your Source Documents ===\n")

# Get a sample document
scraped_df = spark.table(f"`{policy_catalog}`.`{policy_schema}`.authority_scraped")

print(f"Total documents: {scraped_df.count()}\n")

# Get first document
first_doc = scraped_df.first()

print(f"Sample document URL: {first_doc.link}\n")
print(f"Text length: {len(first_doc.scraped_text)} characters\n")
print("=" * 80)
print("First 2000 characters of scraped text:")
print("=" * 80)
print(first_doc.scraped_text[:2000])
print("=" * 80)

# Check for keywords that should be present
keywords = ["QSEHRA", "ICHRA", "HRA", "health reimbursement", "employer", "employee", "requirement", "eligible"]
print("\nKeyword check:")
for keyword in keywords:
    count = first_doc.scraped_text.lower().count(keyword.lower())
    status = "✓" if count > 0 else "✗"
    print(f"  {status} '{keyword}': {count} occurrences")

# COMMAND ----------

# DBTITLE 1,Test Extraction with Sample HRA Text
print("=== Testing Extraction with Known Good Text ===\n")

# Sample text that SHOULD extract entities
test_text = """
Title: Qualified Small Employer Health Reimbursement Arrangement (QSEHRA)
Document Type: IRS Guidance
Citation: 26 USC 9831

The Qualified Small Employer Health Reimbursement Arrangement (QSEHRA) is a type of 
health reimbursement arrangement established by the 21st Century Cures Act in 2016. 
It allows small employers to reimburse employees for medical expenses.

ELIGIBILITY:
Small employers with fewer than 50 full-time employees are eligible to offer a QSEHRA. 
Employees must be offered individual health insurance coverage.

REQUIREMENTS:
1. Written Notice: Employers must provide employees with written notice at least 
   90 days before the beginning of the plan year.
2. Reporting: Employers must report QSEHRA contributions on employees' W-2 forms.
3. Documentation: Employers must maintain records of reimbursements.

FINANCIAL LIMITS:
For 2024, the maximum annual contribution limit is:
- $6,150 for individual coverage
- $12,450 for family coverage
These amounts are indexed for inflation.

TAX TREATMENT:
- Employer contributions are tax-deductible for the employer
- Reimbursements are excludable from employee income if the employee has 
  individual health coverage

PENALTIES:
Failure to provide required written notice may result in an excise tax of $100 
per day per employee, up to $50,000 per year.

STAKEHOLDERS:
- IRS: Regulates and enforces QSEHRA requirements
- DOL: Oversees compliance with notice requirements
- Small Employers: Offer and fund QSEHRAs
- Employees: Receive reimbursements for medical expenses
"""

# Test extraction
print("Testing extraction on sample text...\n")

# Import necessary libraries
import json
from langchain_core.messages import HumanMessage, SystemMessage

try:
    from langchain_databricks import ChatDatabricks
except ImportError:
    from langchain_community.chat_models import ChatDatabricks

# Initialize LLM
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=4000
)

# Use the same prompt from the main script
ENTITY_EXTRACTION_PROMPT = """You are an expert at extracting structured information from healthcare policy documents, specifically about Health Reimbursement Arrangements (HRAs) under the Affordable Care Act (ACA).

Your task is to carefully read the provided policy text and extract ALL relevant entities. Be thorough and precise.

**IMPORTANT INSTRUCTIONS:**
1. Extract ONLY information that is explicitly stated in the text
2. Do NOT make assumptions or infer information not present
3. For IDs, create descriptive, unique identifiers (e.g., "QSEHRA_PLAN", "SMALL_EMPLOYER_STAKEHOLDER")
4. If specific dates, amounts, or details are mentioned, include them
5. Be comprehensive - extract ALL entities of each type that you find

**ENTITY TYPES TO EXTRACT:**
1. **PolicyDocument**: Any law, regulation, code section, guidance, notice, or rule mentioned
2. **HRAPlan**: Types of HRAs (QSEHRA, ICHRA, GCHRA, EBHRA, or other arrangements)
3. **Stakeholder**: Employers, employees, insurers, regulators (IRS, DOL, CMS, HHS), providers, dependents
4. **Requirement**: Things that MUST be done (notices, reporting, documentation, filings)
5. **EligibilityCriteria**: Who can participate or qualify (employee status, employer size, income limits)
6. **Benefit**: Advantages provided (reimbursements, tax credits, deductions, subsidies)
7. **Restriction**: Limitations or prohibitions (coverage limits, participation limits, use restrictions)
8. **Penalty**: Consequences for non-compliance (fines, taxes, disqualifications)
9. **Procedure**: Processes to follow (enrollment, claims, appeals, notifications)
10. **FinancialLimit**: Dollar amounts and thresholds (contribution limits, reimbursement caps)
11. **TaxImplication**: Tax consequences (deductible, excludable, taxable income)
12. **Exception**: Special cases or exemptions to rules
13. **Deadline**: Important dates and timelines

**POLICY TEXT:**

{text}

**OUTPUT:**
Return a JSON object matching the ExtractedEntities schema with all entities you found.
If you don't find any entities of a particular type, return an empty list for that type.
"""

prompt = ENTITY_EXTRACTION_PROMPT.format(text=test_text)

messages = [
    SystemMessage(content="You are a precise entity extraction system. Always return valid JSON."),
    HumanMessage(content=prompt)
]

print("Calling LLM...")
response = llm.invoke(messages)
response_text = response.content

print("\n" + "=" * 80)
print("LLM Response:")
print("=" * 80)
print(response_text[:1000])
print("=" * 80)

# Try to parse the response
import re
json_match = re.search(r'```(?:json)?\s*(.*?)\s*```', response_text, re.DOTALL)
if json_match:
    response_text = json_match.group(1)

try:
    entities = json.loads(response_text)
    print("\n✓ Successfully parsed JSON\n")
    
    print("Entities extracted:")
    total = 0
    for entity_type, entity_list in entities.items():
        count = len(entity_list) if isinstance(entity_list, list) else 0
        total += count
        if count > 0:
            print(f"  ✓ {entity_type}: {count}")
    
    print(f"\nTotal entities: {total}")
    
    if total == 0:
        print("\n⚠️ WARNING: LLM returned empty results even for known good text!")
        print("   This suggests an issue with:")
        print("   1. The LLM model endpoint")
        print("   2. The extraction prompt")
        print("   3. The LLM's capabilities")
        
except Exception as e:
    print(f"\n✗ Error parsing JSON: {e}")
    print("   The LLM may not be returning valid JSON")

# COMMAND ----------

# DBTITLE 1,Test with YOUR Document
print("=== Testing Extraction with YOUR Document ===\n")

# Get your first document
first_doc = scraped_df.first()

print(f"Document: {first_doc.link}")
print(f"Text length: {len(first_doc.scraped_text)} characters\n")

# Truncate if too long
text = first_doc.scraped_text
if len(text) > 8000:
    text = text[:8000] + "\n\n[Text truncated for testing...]"
    print("Note: Text truncated to first 8000 characters\n")

prompt = ENTITY_EXTRACTION_PROMPT.format(text=text)

messages = [
    SystemMessage(content="You are a precise entity extraction system. Always return valid JSON."),
    HumanMessage(content=prompt)
]

print("Calling LLM...")
response = llm.invoke(messages)
response_text = response.content

print("\n" + "=" * 80)
print("LLM Response (first 1000 chars):")
print("=" * 80)
print(response_text[:1000])
print("=" * 80)

# Try to parse
json_match = re.search(r'```(?:json)?\s*(.*?)\s*```', response_text, re.DOTALL)
if json_match:
    response_text = json_match.group(1)

try:
    entities = json.loads(response_text)
    print("\n✓ Successfully parsed JSON\n")
    
    print("Entities extracted from YOUR document:")
    total = 0
    for entity_type, entity_list in entities.items():
        count = len(entity_list) if isinstance(entity_list, list) else 0
        total += count
        if count > 0:
            print(f"  ✓ {entity_type}: {count}")
            # Show first entity as example
            if count > 0 and isinstance(entity_list, list):
                print(f"    Example: {json.dumps(entity_list[0], indent=6)}")
    
    print(f"\nTotal entities: {total}")
    
    if total == 0:
        print("\n⚠️ No entities extracted from your document!")
        print("\nPossible reasons:")
        print("1. Document content doesn't contain HRA/policy information")
        print("2. Scraped text is malformed or just navigation/menu text")
        print("3. LLM model isn't understanding the content")
        print("\nCheck the scraped_text above - does it contain actual policy content?")
        
except Exception as e:
    print(f"\n✗ Error parsing JSON: {e}")

# COMMAND ----------

# DBTITLE 1,Recommendations
print("\n" + "=" * 80)
print("RECOMMENDATIONS")
print("=" * 80)

print("""
Based on the tests above:

IF test with sample text WORKED (extracted entities):
  ✓ LLM and prompt are working
  → Problem is with your SOURCE DOCUMENTS
  → Check if scraped_text contains actual policy content
  → May need to re-scrape or use different URLs

IF test with sample text FAILED (no entities):
  ✗ Issue with LLM or prompt
  → Try different model: databricks-claude-sonnet-4-5 (if available)
  → Try simpler prompt (remove complex instructions)
  → Model may not be capable of this task

IF your document test found SOME entities:
  ✓ System is working but document quality varies
  → Some documents may not have HRA content
  → Filter to only process HRA-related documents

NEXT STEPS:
1. Review the scraped_text from Cell 2 - does it look like policy content?
2. If not, check your step1_scrap_policy.py and source URLs
3. If yes but extraction failed, try different LLM model
4. Consider adding examples to the extraction prompt
""")

